# Actividad Final – Programación con IA (Maestría CDIA, UDI)
## Pipeline de IA para mantenimiento predictivo de una fresadora CNC
**Autora:** María Fernanda Rivera Sanclemente · **Tema:** Automatización de procesos · **Dataset:** AI4I 2020 (UCI)

Este cuaderno ejecuta el pipeline orientado a objetos del paquete `src/pdm_pipeline` y muestra el dashboard.

### 1. Preparar el entorno (clonar repositorio e instalar dependencias)

In [10]:
# Clona el repositorio (en Colab) y entra a la carpeta del proyecto
%cd /content
!rm -rf programacion-ia-udi
!git clone -q https://github.com/mrivera7-max/programacion-ia-udi.git
%cd programacion-ia-udi/actividad_final/actividad_final
!pip install -q -r requirements.txt

/content
/content/programacion-ia-udi/actividad_final/actividad_final


### 2. Ejecutar el pipeline completo
Carga → valida → compara 3 modelos con CV estratificada → ablación → umbral por costo → prueba → importancia → dashboard.

In [12]:
import logging, os, sys

# Ruta absoluta del proyecto en Colab
PROY = "/content/programacion-ia-udi/actividad_final/actividad_final"
os.chdir(PROY)                                   # trabaja dentro del proyecto
sys.path.insert(0, os.path.join(PROY, "src"))    # hace importable el paquete
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

from pdm_pipeline import PipelineConfig, PredictiveMaintenancePipeline

config = PipelineConfig(cost_false_negative=10, cost_false_positive=1)
pipe = PredictiveMaintenancePipeline(config)
resumen = pipe.run()
resumen["cv"]                                    # tabla comparativa de modelos

,model,threshold,roc_auc,pr_auc,precision,recall,f1,tn,fp,fn,tp,expected_cost
1,random_forest,0.34,0.9782,0.8915,0.8712,0.8487,0.8598,7695,34,41,230,444.0
2,gradient_boosting,0.12,0.9662,0.8717,0.8255,0.8376,0.8315,7681,48,44,227,488.0
0,logistic_regression,0.80,0.9223,0.4552,0.3347,0.6199,0.4347,7395,334,103,168,1364.0


In [13]:
# Métricas del mejor modelo en el conjunto de prueba (evaluado una sola vez)
import pandas as pd
pd.Series(resumen["test"])

,0
model,random_forest
threshold,0.34
roc_auc,0.9834
pr_auc,0.8776
precision,0.9048
recall,0.8382
f1,0.8702
tn,1926
fp,6
fn,11


In [14]:
# Estudio de ablación: PR-AUC con y sin variables físicas
resumen["ablation"]

{'con_fisicas': np.float64(0.8915), 'sin_fisicas': np.float64(0.7364)}

### 3. Dashboard interactivo

In [15]:
# Muestra cada figura del dashboard dentro del cuaderno
from pdm_pipeline.dashboard import DashboardBuilder
for fig in DashboardBuilder(pipe).figures():
    fig.show()

### 4. Simulador operativo (Gradio)
Genera un enlace público para evaluar condiciones de operación en tiempo real.

In [17]:
# Lanza el simulador dentro del cuaderno con enlace público
import os, sys
sys.path.insert(0, os.getcwd())          # carpeta del proyecto (fijada en la celda 2)
import app_gradio
app_gradio.build_app().launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://67299e259432343a15.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### 5. Pruebas unitarias

In [18]:
!pytest -q tests

.....                                                                    [100%]
5 passed in 1.62s
